# Scikit-learn homework: algorithms and datasets

Use only NumPy, Matplotlib, the standard library, and scikit-learn. The datasets are in the `data/` folder.

You will build a classification model, a regression model, and two unsupervised mini-exercises.

In [1]:
from pathlib import Path
import csv, math, warnings
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path('data')
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
np.set_printoptions(precision=3, suppress=True)
warnings.filterwarnings('ignore', category=FutureWarning)

def load_mixed_csv(path, target_column):
    with open(path, newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = list(reader)
    target_index = header.index(target_column)
    feature_names = [name for i, name in enumerate(header) if i != target_index]
    X_rows, y_values = [], []
    for row in rows:
        X_rows.append([value for i, value in enumerate(row) if i != target_index])
        y_values.append(row[target_index])
    return np.array(X_rows, dtype=object), np.array(y_values), feature_names

def convert_numeric_columns(X, numeric_indices):
    X = X.copy()
    for idx in numeric_indices:
        X[:, idx] = [np.nan if value == '' else float(value) for value in X[:, idx]]
    return X

def print_classification_scores(y_true, y_pred):
    from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
    print('accuracy:', accuracy_score(y_true, y_pred))
    print('f1:', f1_score(y_true, y_pred))
    print('confusion matrix:')
    print(confusion_matrix(y_true, y_pred))

def print_regression_scores(y_true, y_pred):
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    print('MAE:', mean_absolute_error(y_true, y_pred))
    print('RMSE:', math.sqrt(mean_squared_error(y_true, y_pred)))
    print('R2:', r2_score(y_true, y_pred))

In [22]:
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Part A — Classification: student success

Dataset: `data/student_success.csv`. Target: `passed`.

## Task A1
Load the dataset. Print feature names with index numbers, shapes, first 3 rows, and class counts.

In [11]:
X_raw, y_raw, features_names = load_mixed_csv(DATA_DIR / 'student_success.csv', 'passed')
y = y_raw.astype(int)

for i, name in enumerate(features_names):
    print(f"feature {i}: {name}")

print(f"X shape is {X_raw.shape}")
print(f"y shape is {y.shape}")
print(f"First three rows is: {X_raw[:3]}")
classes, counts = np.unique(y, return_counts=True)
print(f"Class counts: {dict(zip(classes.tolist(), counts.tolist()))}")

feature 0: study_hours
feature 1: attendance_pct
feature 2: previous_score
feature 3: sleep_hours
feature 4: practice_tests
feature 5: course_level
feature 6: internet_access
feature 7: part_time_job
X shape is (440, 8)
y shape is (440,)
First three rows is: [['21.3' '100.0' '98.4' '5.5' '7' 'regular' 'stable' 'no']
 ['4.8' '70.3' '70.8' '7.6' '2' 'regular' '' 'no']
 ['13.2' '85.7' '' '7.0' '3' 'advanced' 'unstable' 'yes']]
Class counts: {0: 109, 1: 331}


## Task A2
Define numeric and categorical columns by index.

Numeric: `study_hours`, `attendance_pct`, `previous_score`, `sleep_hours`, `practice_tests`.

Categorical: `course_level`, `internet_access`, `part_time_job`.

Convert numeric columns with `convert_numeric_columns`.

In [20]:
numeric_cols = [features_names.index(name) for name in ["study_hours", "attendance_pct", "previous_score", "sleep_hours", "practice_tests"]]
categorical_cols = [features_names.index(name) for name in ["course_level", "internet_access", "part_time_job"]]

X = convert_numeric_columns(X_raw, numeric_cols)

print(f"numeric_cols : {numeric_cols}")
print(f"cate_cols: {categorical_cols}")
print(f"Converted first row is {X[0]}")


numeric_cols : [0, 1, 2, 3, 4]
cate_cols: [5, 6, 7]
Converted first row is [21.3 100.0 98.4 5.5 7.0 'regular' 'stable' 'no']


## Task A3
Create a stratified train/test split with `test_size=0.25` and `random_state=RANDOM_STATE`.

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

## Task A4
Build preprocessing with `SimpleImputer`, `StandardScaler`, `OneHotEncoder`, and `ColumnTransformer`.

In [26]:
numeric_pre = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())
])

categorical_pre = Pipeline([
    ('imputer', SimpleImputer(missing_values='', strategy="most_frequent")),
    ('onehot', OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer([
    ('num', numeric_pre, numeric_cols),
    ('cat', categorical_pre, categorical_cols)
])

## Task A5
Fit a logistic regression baseline pipeline and evaluate it with accuracy, F1, and a confusion matrix.

In [27]:
from sklearn.linear_model import LogisticRegression
logistic_pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=1000))
])

logistic_pipe.fit(X_train, y_train)
y_pred = logistic_pipe.predict(X_test)
print_classification_scores(y_test, y_pred)

accuracy: 0.7909090909090909
f1: 0.8700564971751412
confusion matrix:
[[10 17]
 [ 6 77]]


## Task A6
Compare at least six classifiers with 4-fold stratified cross-validation: logistic regression, kNN, decision tree, random forest, SVM, and GaussianNB.

In [28]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

"sklearn.model_selection.cross_validate(estimator, X, y=None, *, groups=None, scoring=None, cv=None, n_jobs=None, verbose=0, params=None, pre_dispatch='2*n_jobs', return_train_score=False, return_estimator=False, return_indices=False, error_score=nan)"

In [34]:
classifiers = {
    'logistic_regression': LogisticRegression(max_iter=1000),
    'random_forest': RandomForestClassifier(n_estimators=80),
    'gaussian_nb': GaussianNB(),
    'svm_rbf': SVC(kernel='rbf', C=1.0)
}

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
algs = []

for name, model in classifiers.items():
    pipeee = Pipeline([
        ('preprocess', preprocess),
        ('model', model)
    ])
    scores = cross_validate(pipeee, X_train, y_train, scoring=['accuracy','f1'])
    # print(scores)
    algs.append((name, scores["test_accuracy"].mean(), scores["test_f1"].mean()))

algs

[('logistic_regression',
  np.float64(0.7939393939393941),
  np.float64(0.8699236088096927)),
 ('random_forest',
  np.float64(0.7575757575757576),
  np.float64(0.8479655941143178)),
 ('gaussian_nb',
  np.float64(0.7727272727272727),
  np.float64(0.8510442832049924)),
 ('svm_rbf', np.float64(0.7787878787878788), np.float64(0.8670682895899887))]

## Task A7
Tune one model with `GridSearchCV`, scoring by F1. Print best parameters, best CV F1, and final test scores.

In [35]:
from sklearn.model_selection import GridSearchCV

In [36]:
param_grid = {
    'model__n_estimators': [60, 120],
    'model__max_depth': [4, None],
    'model__min_samples_leaf': [1, 6],
}

random_f = Pipeline([
    ('preprocess', preprocess),
    ('model', RandomForestClassifier())
])

grid = GridSearchCV(random_f, param_grid=param_grid, scoring='f1', cv=cv)
grid.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=4, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scalar',
                                                                                          StandardScaler())]),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4]),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(missing_values='',
                                                                                                        strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore',
                                                                                                        sparse_output=False))]),
                                                                         [5, 6,
                                                                          7])])),
                                       ('model', RandomForestClassifier())]),
             param_grid={'model__max_depth': [4, None],
                         'model__min_samples_leaf': [1, 6],
                         'model__n_estimators': [60, 120]},
             scoring='f1')

In [37]:
grid.best_score_

np.float64(0.8746731621891706)

In [38]:
grid.best_params_

{'model__max_depth': None,
 'model__min_samples_leaf': 6,
 'model__n_estimators': 120}

# Part B — Regression: used-car price

Dataset: `data/used_cars.csv`. Target: `price_eur`.

## Task B1
Load and inspect the data.

In [ ]:
# Your code here

## Task B2
Build preprocessing.

Numeric: `year`, `mileage`, `engine_size_l`.

Categorical: `fuel`, `transmission`, `brand_tier`.

In [ ]:
# Your code here

## Task B3
Compare Ridge, Lasso, RandomForestRegressor. Report MAE, RMSE, and R².

In [ ]:
# Your code here

## Task B4
Tune a RandomForestRegressor using `GridSearchCV` with `scoring='neg_mean_absolute_error'`.

In [ ]:
# Your code here

# Part C — Unsupervised learning

## Task C1
Create a `make_blobs` dataset with 500 rows and 4 centers. Scale it, fit KMeans, and plot clusters.

In [ ]:
from sklearn.datasets import make_blobs, load_digits
X_blob, _ = make_blobs(n_samples=500, centers=4, cluster_std=1.4, random_state=RANDOM_STATE)

## Task C2
Load the digits dataset, scale it, reduce to 2 PCA components, and plot colored by digit label.

In [ ]:
# Your code here

## Final reflection
Write 3–6 sentences: which model you would choose, which metric matters most, and where leakage could happen without pipelines.